# INFO8665 Local Orchestrator Notebook

This notebook orchestrates the **local-first** workflow:

1. Download Roboflow Universe datasets (optional; requires `ROBOFLOW_API_KEY`).
2. Merge/normalize banknotes into a canonical detector dataset.
3. Build a negatives pool.
4. Build bill cutouts (split-preserving ImageFolder).
5. Lightweight EDA summary.
6. Train locally (detector + coin classifier + bill classifier).
7. Export artifacts (ONNX / optional INT8) and save quantized models locally.

All outputs are written under `data/` and `outputs/` (both are git-ignored).

## 0) Setup

This notebook expects to be run from the INFO8665 repo root.

- Local Python: `pip install -r requirements.txt`
- Docker: `docker build -t info8665-local .`

Roboflow download requires an API key:

- PowerShell: `$env:ROBOFLOW_API_KEY = "..."`
- bash: `export ROBOFLOW_API_KEY="..."`

The pipeline still works without the download step if you already have `data/interim/...` populated.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from subprocess import run

REPO_ROOT = Path.cwd()
print("cwd=", REPO_ROOT)


def sh(cmd: list[str]) -> None:
    print("\n$", " ".join(cmd))
    run(cmd, check=False)


def has_env(name: str) -> bool:
    return bool(os.environ.get(name, "").strip())


print("has_ROBOFLOW_API_KEY=", has_env("ROBOFLOW_API_KEY"))

## 1) Download from Roboflow Universe (optional)

Downloads to `data/interim/<category>/...`.

Skip this section if you already have datasets under `data/interim/`.

In [ ]:
# Requires ROBOFLOW_API_KEY; this will exit if not set.
# Comment out if you want to skip download.
sh([
    sys.executable,
    "scripts/01_download_from_universe.py",
    "--links",
    "configs/universe_links.yaml",
    "--out",
    "data/interim",
    "--format",
    "yolov8",
])

## 2) Merge + normalize datasets

### 2a) Banknotes (for bill cutouts + bill classifier)

- Output: `data/processed/banknotes_merged/`
- Config: `configs/label_map.yaml`

### 2b) Money detector dataset (banknotes + coins)

If you want the detector to detect **banknotes and coins**, build:

- Output: `data/processed/money_merged/`
- Config: `configs/label_map_money.yaml` (adds `COIN`)

In [ ]:
# 2a) Banknotes merged
sh([
    sys.executable,
    "scripts/02_merge_and_normalize_banknotes.py",
    "--in-root",
    "data/interim/banknotes",
    "--label-map",
    "configs/label_map.yaml",
    "--out",
    "data/processed/banknotes_merged",
])

# 2b) Money merged (banknotes + coins) for a unified detector
sh([
    sys.executable,
    "scripts/02_merge_and_normalize_money.py",
    "--banknotes-root",
    "data/interim/banknotes",
    "--coins-root",
    "data/interim/coins",
    "--label-map",
    "configs/label_map_money.yaml",
    "--out",
    "data/processed/money_merged",
])

## 3) Negatives pool (confusers + backgrounds)

Build a deduplicated negative image pool:

- Output: `data/processed/negatives/images/`

In [ ]:
sh([
    sys.executable,
    "scripts/03_build_negatives_pool.py",
    "--confusers-root",
    "data/interim/confusers",
    "--backgrounds-root",
    "data/interim/backgrounds",
    "--out",
    "data/processed/negatives",
])

## 4) Build bill cutouts (split-preserving)

Creates RGBA PNG cutouts for bill classification:

- Output: `data/processed/bill_cutouts/{train,val,test}/<class>/*.png`

In [ ]:
sh([
    sys.executable,
    "scripts/04_make_bill_cutouts.py",
    "--dataset",
    "data/processed/banknotes_merged",
    "--out",
    "data/processed/bill_cutouts",
])

## 5) Lightweight EDA summary

Writes a small JSON report with split counts + class balance.

In [ ]:
sh([
    sys.executable,
    "scripts/06_eda_banknotes_summary.py",
    "--dataset",
    "data/processed/banknotes_merged",
    "--out-json",
    "outputs/reports/banknotes_eda_summary.json",
])

## 6) Train detector (YOLO26)

Trains locally using Ultralytics. Outputs under `outputs/models/<run-name>/...`.

In [ ]:
# Prefer the unified money_merged dataset if present; fall back to banknotes_merged.
money_yaml = Path("data/processed/money_merged/data.yaml")
banknotes_yaml = Path("data/processed/banknotes_merged/data.yaml")

train_yaml = str(money_yaml) if money_yaml.exists() else str(banknotes_yaml)
print("detector_data=", train_yaml)

sh([
    sys.executable,
    "scripts/10_train_detector.py",
    "--data",
    train_yaml,
    "--weights",
    "yolo26n.pt",
    "--imgsz",
    "640",
    "--epochs",
    "25",
    "--batch",
    "16",
    "--project",
    "outputs/models",
    "--run-name",
    "detector_local",
])

## 7) Export detector to ONNX (optional INT8)

- Standard ONNX: does not require calibration
- INT8 ONNX: requires calibration dataset (`--data`) and may take time

In [ ]:
# Export standard ONNX (no calibration)
# For INT8 calibrated ONNX, add: --int8 --data data/processed/banknotes_merged/data.yaml

trained_weights = Path("outputs/models/detector_local/weights/best.pt")
weights_for_export = str(trained_weights) if trained_weights.exists() else "yolo26n.pt"
print("weights_for_export=", weights_for_export)

sh([
    sys.executable,
    "scripts/11_export_detector_onnx.py",
    "--weights",
    weights_for_export,
    "--imgsz",
    "640",
    "--out",
    "outputs/models/detector_export.onnx",
])

## 8) Train coin classifier (local)

Expects ImageFolder train/val folders (provide your own paths, or generate from your coin crop workflow).

In [ ]:
# Replace these with your prepared ImageFolder paths
coin_train = "data/processed/coin_domains/A_synth"  # example placeholder
coin_val = "data/processed/coin_domains/B_real"     # example placeholder

# Train
sh([
    sys.executable,
    "scripts/22_train_coin_classifier.py",
    "--train-dir",
    coin_train,
    "--val-dir",
    coin_val,
    "--output",
    "outputs/models/coin_classifier_local.pt",
    "--epochs",
    "20",
    "--batch-size",
    "64",
    "--lr",
    "0.001",
    "--img-size",
    "256",
    "--backbone",
    "resnet18",
])

## 9) Train bill classifier (local)

Uses the split-preserving bill cutouts from step 4.

In [ ]:
sh([
    sys.executable,
    "scripts/24_train_bill_classifier.py",
    "--train-dir",
    "data/processed/bill_cutouts/train",
    "--val-dir",
    "data/processed/bill_cutouts/val",
    "--output",
    "outputs/models/bill_classifier_local.pt",
    "--epochs",
    "20",
    "--batch-size",
    "64",
    "--lr",
    "0.001",
    "--img-size",
    "256",
    "--backbone",
    "resnet18",
])

## 10) Save quantized models locally (TorchScript)

Creates CPU-friendly TorchScript artifacts using dynamic INT8 quantization on Linear layers.

In [ ]:
# Coin classifier quantized TorchScript
sh([
    sys.executable,
    "scripts/26_quantize_classifier_torchscript.py",
    "--checkpoint",
    "outputs/models/coin_classifier_local.pt",
    "--out",
    "outputs/models/coin_classifier_quantized.ts.pt",
    "--meta-out",
    "outputs/models/coin_classifier_quantized.meta.json",
])

# Bill classifier quantized TorchScript
sh([
    sys.executable,
    "scripts/26_quantize_classifier_torchscript.py",
    "--checkpoint",
    "outputs/models/bill_classifier_local.pt",
    "--out",
    "outputs/models/bill_classifier_quantized.ts.pt",
    "--meta-out",
    "outputs/models/bill_classifier_quantized.meta.json",
])

## Notes

- Detector training in this repo is currently set up for *banknotes* (CAD_5/10/20/50/100) based on `configs/label_map.yaml`.
- Coin handling is via a **coin classifier** trained on cropped coin images; a separate *coin detector* is not included in this INFO8665 pipeline yet.
- Sprint0 `training/* --smoke` scaffolds are still available for demo/testing without ML dependencies.
- `scripts/*` reflects the local methodology (detector + coin classifier + bill classifier + exports).
- `data/` and `outputs/` are intentionally not committed.

## Optional: download trained champions from AWS

If you have AWS credentials configured locally, you can download SageMaker TrainingJob artifacts:

- `python scripts/40_download_sagemaker_champions.py --region us-east-1 --job <TRAINING_JOB_NAME>`

This writes `outputs/models/aws_champions/<job>/model.tar.gz` and (by default) flattens common files into `outputs/models/`.

In [ ]:
# Quick smoke check (no datasets required)
sh([sys.executable, "training/train_detector.py", "--smoke"])
sh([sys.executable, "training/eval_detector.py", "--smoke"])
sh([sys.executable, "training/coin_classifier/train_coin_classifier.py", "--smoke"])
sh([sys.executable, "training/coin_classifier/eval_coin_classifier.py", "--smoke"])
sh([sys.executable, "training/coin_classifier/infer_coin_classifier.py", "--smoke"])

# Optional: download champions from AWS (requires AWS credentials)
# sh([sys.executable, "scripts/40_download_sagemaker_champions.py", "--job", "champ-detector-YYYYMMDD-HHMMSS", "--region", "us-east-1"])

## (Optional) List outputs

Shows what the pipeline generated locally.

In [ ]:
for path in [Path("data"), Path("outputs")]:
    if not path.exists():
        print(f"missing={path}")
        continue
    print(f"\n{path}/")
    for p in sorted(path.rglob("*")):
        if p.is_file():
            print(p.as_posix())